# Musanga V4 — NB04 V4.002: multispectral quality & relative spectral reconnaissance

**Purpose:** determine which UAV MS data can contribute to Musanga segmentation before any supervised MS model. Reuses NB00 registered MS inventory and RGB survey pairing. **No flight-time metadata assumptions, no GPU, no giant raster reads.**

**Strict policy:** raw/scaled DN are **not** treated as calibrated reflectance; no NDVI/NDRE until band identities, scales and offsets are verified. ROI coverage is **actual valid pixels**, not just raster bounding boxes. Per-survey quality and edge effects are reported separately. Yangambi right-hand MS acquisitions are **not** silently paired with the entire Carlier RGB mosaic; Yoko MS is not presumed to exist.

This notebook computes sparse coordinate/ROI samples, zero/nodata and saturation proxies, image-border distance, RGB correspondence metadata, and exploratory *relative* band contrast. It never edits source imagery. It does not infer cloud vs shadow causation without acquisition-level evidence.

**Review gates:** (1) band identities, (2) radiometric scaling, (3) valid coverage and border QA, (4) registration, (5) shadow/seam QA. All thresholds are diagnostic rather than automatic acceptance.


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_001'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
MAX_PREVIEW=750          # bounded 750x750 raster sampling, per survey/band
MAX_RASTER_PIXEL=600000  # bounded sample point count, never full-res reads
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


In [ ]:
# 01 — Survey/file headers, band descriptions, zero/nodata, bounded native-grid sampling
# Do not infer band identity for multiband stacks with missing descriptions.
from collections import defaultdict
from rasterio.windows import bounds as window_bounds
def histogram_sample(ds,band=1,maxside=MAX_PREVIEW):
    factor=max(1,math.ceil(max(ds.width,ds.height)/maxside))
    width=max(1,math.ceil(ds.width/factor));height=max(1,math.ceil(ds.height/factor))
    # Rasterio samples the ENTIRE extent, without fetching full native-resolution array.
    a=ds.read(band,out_shape=(height,width),resampling=Resampling.nearest,masked=True)
    z=np.asarray(a.data,dtype=np.float64)
    valid=(~np.ma.getmaskarray(a)) & np.isfinite(z)
    return z,valid,(factor,height,width)
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.open(path) as ds:
        for b in range(1,ds.count+1):
            z,valid,_=histogram_sample(ds,b)
            values=z[valid]
            lo,hi=(np.percentile(values,[1,99]) if len(values) else (np.nan,np.nan))
            rows.append(dict(group=rec.group,file=path.name,path=str(path),band=b,
                band_hint=rec.band_hint,
                description=ds.descriptions[b-1] if ds.descriptions else None,
                colorinterp=str(ds.colorinterp[b-1]) if ds.colorinterp else None,
                dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                nodata=ds.nodata,crs=str(ds.crs),width=ds.width,height=ds.height,
                resolution_x=abs(ds.transform.a),resolution_y=abs(ds.transform.e),
                valid_sample_fraction=float(valid.mean()),n_sample=int(valid.size),
                p01=float(lo),p50=float(np.median(values)) if len(values) else np.nan,
                p99=float(hi),min=float(values.min()) if len(values) else np.nan,
                max=float(values.max()) if len(values) else np.nan,
                zero_fraction_valid=float(np.mean(values==0)) if len(values) else np.nan,
                near_top_fraction=float(np.mean(values>=hi)) if len(values) else np.nan,
                file_bytes=path.stat().st_size))
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty,'No readable MS rasters.'
BAND_AUDIT.to_csv(OUT/'04_band_header_distribution_audit.csv',index=False)
display(BAND_AUDIT[['group','file','band','description','band_hint',
                    'dtype','scale','offset','nodata','valid_sample_fraction',
                    'p01','p50','p99','zero_fraction_valid']])
print('Important: p99 tails and zeros are diagnostics, not automatic cloud/shadow masks.')


In [ ]:
# 02 — FAST sparse spatial QA: ~100 points per mosaic, no full warped raster
# Purpose: approximate ROI/footprint validity, not exact pixel coverage.
from rasterio.warp import transform as transform_xy
from rasterio.transform import xy as pixel_xy
GRID_N=10
summary=[];qa_masks={}
for group,rec_group in ms.groupby('group',sort=True):
    if group not in RGB_FILES: continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        h=w=GRID_N
        tr=from_bounds(*rgb.bounds,w,h)
        xx,yy=np.meshgrid(np.arange(w),np.arange(h))
        xs,ys=pixel_xy(tr,yy.ravel(),xx.ravel(),offset='center')
        xs=np.asarray(xs);ys=np.asarray(ys)
        local_roi=roi.to_crs(rgb.crs).geometry
        inroi=rasterize(((g,1) for g in local_roi if g is not None and not g.is_empty),
                        out_shape=(h,w),transform=tr,fill=0,dtype='uint8').astype(bool)
        # Sparse RGB validity: no resampling of the huge mosaic.
        rgb_points=list(rgb.sample(zip(xs,ys),indexes=[1,2,3],masked=True))
        rgb_valid=np.array([
            np.isfinite(np.asarray(v.data,dtype=float)).all()
            and not np.ma.getmaskarray(v).any() for v in rgb_points
        ]).reshape(h,w)
        roi_ref=inroi & rgb_valid
        print('SAMPLING',group,'ROI points',int(roi_ref.sum()),'/',GRID_N*GRID_N,flush=True)
        for rec in rec_group.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.open(path) as src:
                if src.crs is None:continue
                if src.crs==rgb.crs:
                    mx,my=xs,ys
                else:
                    mx,my=transform_xy(rgb.crs,src.crs,xs.tolist(),ys.tolist())
                # Exact coordinate samples at 100 locations; no WarpedVRT and
                # no full-extent read. NaN/nodata checked; 2nd band unclassified.
                samples=list(src.sample(zip(mx,my),indexes=1,masked=True))
                val=np.array([float(v.data[0]) for v in samples],np.float32).reshape(h,w)
                valid=np.array([bool(not np.ma.getmaskarray(v)[0]) and
                                 np.isfinite(float(v.data[0])) for v in samples]).reshape(h,w)
                support=valid & roi_ref
                fraction=float(support.sum()/max(1,roi_ref.sum()))
                key=f'{group}_{rec.name}'
                # Not a metric distance-to-valid-edge map! Only an approximate
                # distance to raster rectangular extent for sampled ROI points.
                left,bottom,right,top=transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21)
                nominal_extent_dist=np.minimum.reduce([xs-left,right-xs,ys-bottom,top-ys])
                edge=np.maximum(nominal_extent_dist,0).reshape(h,w).astype(np.float32)
                qa_masks[key]=(val,support,valid,edge,roi_ref)
                hist={f'fraction_support_bbox_edge_ge_{d}m':
                    float((support&(edge>=d)).sum()/max(1,roi_ref.sum()))
                    for d in EDGE_DISTANCE_M}
                summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                    nominal_bands=int(src.count),roi_valid_fraction_sample=fraction,
                    sampling_grid_points=GRID_N*GRID_N,n_roi_samples=int(roi_ref.sum()),
                    n_valid_roi_samples=int(support.sum()),**hist))
                print(' ',rec.name,'sample valid',int(support.sum()),'/',int(roi_ref.sum()),flush=True)
SUMMARY=pd.DataFrame(summary)
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_valid_coverage_overview.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('IMPORTANT: sparse 10x10 samples are a reconnaissance only, NOT a precise coverage'
      ' estimate. Bbox-edge distance is NOT distance to internal valid-data seams.'
      ' Follow up with small native raster windows at selected locations.')


In [ ]:
# 03 — Fast sparse survey QA figures (not pixel-accurate coverage maps)
for key,(z,support,valid,edge,roi_ref) in qa_masks.items():
    vals=z[support]
    if not vals.size:continue
    p1,p99=np.percentile(vals,[1,99])
    if p99<=p1:p99=p1+1
    fig,axs=plt.subplots(1,3,figsize=(11.5,3))
    axs[0].imshow(np.where(support,z,np.nan),vmin=p1,vmax=p99,
                  cmap='gray',interpolation='nearest')
    axs[0].set_title('Band 1, sparse 10x10 samples')
    axs[1].imshow(support,cmap='gray',vmin=0,vmax=1,interpolation='nearest')
    axs[1].set_title('Sampled support (NOT full coverage)')
    im=axs[2].imshow(np.where(roi_ref,np.minimum(edge,50),np.nan),
                     vmin=0,vmax=50,cmap='viridis',interpolation='nearest')
    axs[2].set_title('Distance to raster bbox (not seam)')
    fig.colorbar(im,ax=axs[2],fraction=.04,label='metres')
    for ax in axs:ax.axis('off')
    fig.suptitle(key,fontsize=9)
    fig.tight_layout()
    safe=re.sub(r'[^0-9A-Za-z_-]+','_',key)[:125]
    fig.savefig(OUT/f'04_qa_{safe}.png',dpi=110,bbox_inches='tight')
    plt.show()
    plt.close(fig)
print('Sparse QA figures saved:',OUT)


## Interpretation and NB04 next stage

At this stage, **do not automatically mask dark pixels as clouds**, discard a uniform fixed border, or normalize unrelated flights to the same spectral distribution.

After inspecting the band summary and maps:
1. Confirm individual band identities (including suspect MOS red/red-edge files) and reflectance scaling; flag ambiguous stacks.
2. Identify which surveys have co-registered *matched* four-band support inside the ROI.
3. On confirmed bands, compute brightness/shadow proxies, local normalized differences and ratios; compare spectra in nearby canopy regions.
4. Inspect RGB/MS alignment in representative Musanga crowns and at image edges.
5. Only after QA, train a small spatially validated relative-spectral classifier on trusted pixels, with RGB fallback where MS is missing.

**Critical:** The NB00 RGB Yangambi reference is not evidence that YBJoseph or YBForgeo MS is co-temporal with the entire Yangambi RGB mosaic. Handle each acquisition footprint independently and confirm spatial alignment. No MS information for Yoko may be invented.
